<a href="https://colab.research.google.com/github/Feellived/molecular-reliability-signals/blob/yoonsoo/A3_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A-3. 파이프라인 2~8단계 (2026-09-26)

`variants_v2`(A-3 변형, 22종 620,806건)를 채점하고 평가한다.
**생성만 끝난 상태이므로 A-3 성공 기준이 아직 판정되지 않았다.**

| 성공 기준 | 현재 |
|---|---|
| 축 기여가 AUPRC +0.0225 를 넘는가 | 미판정 |
| B3 기여가 0에서 벗어나는가 (현재 정규화 AURC 0.984) | 미판정 |

## 단계

```
2) 지문 채점      score_variants_fingerprint.py   -> scores_v2/fingerprint
3) 언어모델 채점   score_variants_chemberta.py     -> scores_v2/chemberta
4) 축 분산        compute_ab_variance.py          -> scores_v2/signals
5) 신호 통합      assemble_signals.py             -> scores_v2/evaluation
6) 조건부 B       build_conditional_signals.py    -> evaluation 에 열 추가
7) 확장 통계      build_rich_variant_features.py  -> evaluation 에 열 추가
8) 판정          run_preregistered_ablation.py   -> 표제 수치
```

## 방침

- **모든 스크립트를 원문 그대로 사용한다.** 로직을 고치지 않는다
- **A축 지문 채점을 생략하지 않는다.** 생략은 제안 단계이고 합의되지 않았다.
  오히려 전부 채점해 A축 분산이 0으로 나오는지 확인하면
  **A축 정준형 왕복 검사가 제대로 작동했는지 검증**할 수 있다
- 출력은 `Yoonsoo/scores_v2/` 로 분리한다. 기존 `scores_role4` 를 덮지 않는다

## 이 노트북이 다루는 범위

2·3단계(채점)를 완전히 수행한다. **채점이 전체 시간의 대부분**이므로 먼저 돌린다.

4~8단계는 스크립트 인자를 확인해야 한다. 특히 **8단계는 담당4의 로컬 절대경로가
하드코딩돼 있어** 그대로 실행되지 않는다. 셀에서 `--help` 와 경로 설정부를 출력하니
그 결과로 마무리한다.

In [1]:
# ============================================================================
# [셀 1] 설치
# ============================================================================
%pip install -q rdkit transformers scikit-learn xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 36.5 MB/s eta 0:00:00


In [2]:
# ============================================================================
# [셀 2] 마운트 + 경로
# ============================================================================
import warnings; warnings.filterwarnings("ignore")
import json, re, subprocess, sys, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

TEAM = Path("/content/drive/MyDrive/Conference_2026")
MINE = TEAM / "Yoonsoo"

VARIANTS_DIR    = MINE / "variants_v2"                                      # A-3 변형
SPLITS_ROOT     = TEAM / "Juhyeong" / "data" / "processed" / "pipeline_yoonsoo"
CHECKPOINT_ROOT = TEAM / "Jiye" / "checkpoints" / "chemberta_seed_42" / "checkpoints"
REFERENCE_DIR   = TEAM / "Jiye" / "outputs"                                 # 담당2 예측 (지문 채점용)

SCORES   = MINE / "scores_v2"
FP_DIR   = SCORES / "fingerprint"
CB_DIR   = SCORES / "chemberta"
SIG_DIR  = SCORES / "signals"
EVAL_DIR = SCORES / "evaluation"

WORK = Path("/content/a3")
WORK.mkdir(parents=True, exist_ok=True)

for label, p in [("변형 입력", VARIANTS_DIR), ("splits", SPLITS_ROOT),
                 ("체크포인트", CHECKPOINT_ROOT), ("담당2 예측", REFERENCE_DIR)]:
    print(f"{label:10s} {'있음' if p.exists() else '없음 <- 확인'}  {p}")

datasets = sorted(p.parent.name for p in VARIANTS_DIR.glob("*/variants.csv"))
print(f"\n대상 물성 {len(datasets)}종")

assert VARIANTS_DIR.is_dir(), f"변형 입력 없음: {VARIANTS_DIR}"
assert datasets, f"variants.csv 없음: {VARIANTS_DIR}"

Mounted at /content/drive
변형 입력      있음  /content/drive/MyDrive/Conference_2026/Yoonsoo/variants_v2
splits     있음  /content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo
체크포인트      있음  /content/drive/MyDrive/Conference_2026/Jiye/checkpoints/chemberta_seed_42/checkpoints
담당2 예측     있음  /content/drive/MyDrive/Conference_2026/Jiye/outputs

대상 물성 22종


In [3]:
# ============================================================================
# [셀 3] 변형 규모 확인
# ============================================================================
rows = []
for ds in datasets:
    v = pd.read_csv(VARIANTS_DIR / ds / "variants.csv", usecols=["axis", "parent_row_uid"])
    c = v.axis.value_counts().to_dict()
    c.update({"dataset": ds, "분자": v.parent_row_uid.nunique(), "행": len(v)})
    rows.append(c)

t = pd.DataFrame(rows)
cols = [c for c in ["dataset", "분자", "행", "A", "B1_tautomer", "B1_protonation", "B3_stereo"]
        if c in t.columns]
print(t[cols].to_string(index=False))
print(f"\n합계 행 {t['행'].sum():,} | 분자 {t['분자'].sum():,}")

                       dataset   분자      행     A  B1_tautomer  B1_protonation  B3_stereo
                          ames 1450  50446 40782         5162            3878      624.0
                   bbb_martins  390  15655 11503         1861            1745      546.0
            bioavailability_ma  127   5222  3800          537             626      259.0
                    caco2_wang  180   7960  5393         1199            1016      352.0
       clearance_hepatocyte_az  203   8570  6090         1049            1261      170.0
        clearance_microsome_az  220   9314  6600         1167            1400      147.0
cyp2c9_substrate_carbonmangels  133   5589  3961          687             610      331.0
                  cyp2c9_veith 2409  97029 72074        11088           11958     1909.0
cyp2d6_substrate_carbonmangels  132   5407  3902          635             563      307.0
                  cyp2d6_veith 2616 105921 78311        12478           12897     2235.0
cyp3a4_substrate_carb

## 1단계 — 스크립트 확보

`rglob` 은 드라이브 바로가기를 통과하지 못하므로 `iterdir` 재귀로 찾는다.
**내용은 고치지 않고 그대로 복사한다.**

In [5]:
# ============================================================================
# [셀 4] 스크립트 복사 (수정 없음)
# ----------------------------------------------------------------------------
# 전부 Main/scripts_role4/ 에 있음이 확인돼 있어 직접 복사한다. 탐색하지 않는다.
# ============================================================================
SRC_DIR = TEAM / "Main" / "scripts_role4"

NEEDED = [
    "score_variants_fingerprint.py",
    "score_variants_chemberta.py",
    "compute_ab_variance.py",
    "assemble_signals.py",
    "build_conditional_signals.py",
    "build_rich_variant_features.py",
    "run_preregistered_ablation.py",
    "dataset_repairs.py",
]

print("원본:", SRC_DIR, "(있음)" if SRC_DIR.exists() else "(없음)")
missing = []
for name in NEEDED:
    src = SRC_DIR / name
    if src.exists():
        shutil.copy2(src, WORK / name)
        print(f"[복사] {name}")
    else:
        missing.append(name)
        print(f"[없음] {name}")

print("\n작업 폴더:", sorted(p.name for p in WORK.glob("*.py")))
assert not missing, f"다음 파일을 못 찾았다: {missing}"

원본: /content/drive/MyDrive/Conference_2026/Main/scripts_role4 (있음)
[복사] score_variants_fingerprint.py
[복사] score_variants_chemberta.py
[복사] compute_ab_variance.py
[복사] assemble_signals.py
[복사] build_conditional_signals.py
[복사] build_rich_variant_features.py
[복사] run_preregistered_ablation.py
[복사] dataset_repairs.py

작업 폴더: ['assemble_signals.py', 'build_conditional_signals.py', 'build_rich_variant_features.py', 'compute_ab_variance.py', 'dataset_repairs.py', 'run_preregistered_ablation.py', 'score_variants_chemberta.py', 'score_variants_fingerprint.py']


## 2단계 — 지문 채점

담당2 모델을 물성마다 재적합해 변형을 채점한다. **A축도 포함해 전부 채점한다.**

In [6]:
# ============================================================================
# [셀 5] 지문 채점 (진행 상황 실시간 출력)
# ============================================================================
cmd = (f'cd {WORK} && python score_variants_fingerprint.py'
       f' --processed-dir "{SPLITS_ROOT}"'
       f' --variants-dir "{VARIANTS_DIR}"'
       f' --reference-dir "{REFERENCE_DIR}"'
       f' --out-dir "{FP_DIR}"'
       f' --resume')
print(cmd, "\n")
!{cmd}

cd /content/a3 && python score_variants_fingerprint.py --processed-dir "/content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo" --variants-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/variants_v2" --reference-dir "/content/drive/MyDrive/Conference_2026/Jiye/outputs" --out-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/fingerprint" --resume 

[1/22] ames: 원본 7,255 + 변형 50,446 채점 (178.0초, 대표 rf, 원본 대비 최대차 0.000)
[2/22] bbb_martins: 원본 1,955 + 변형 15,655 채점 (45.2초, 대표 rf, 원본 대비 최대차 0.000)
[3/22] bioavailability_ma: 원본 640 + 변형 5,222 채점 (21.2초, 대표 rf, 원본 대비 최대차 0.000)
[4/22] caco2_wang: 원본 905 + 변형 7,960 채점 (33.4초, 대표 xgb, 원본 대비 최대차 0.000)
[5/22] clearance_hepatocyte_az: 원본 1,020 + 변형 8,570 채점 (34.3초, 대표 rf, 원본 대비 최대차 0.000)
[6/22] clearance_microsome_az: 원본 1,102 + 변형 9,314 채점 (35.5초, 대표 xgb, 원본 대비 최대차 0.000)
[7/22] cyp2c9_substrate_carbonmangels: 원본 666 + 변형 5,589 채점 (22.2초, 대표 xgb, 원본 대비 최대차 0.000)
[8/22] cyp2c9_veith: 원본 12,047 + 변형 97,029 

In [9]:
# half_life_obach만 다시 채점
import shutil
out = FP_DIR / "half_life_obach"
if out.exists():
    shutil.rmtree(out)

cmd = (f'cd {WORK} && python -u score_variants_fingerprint.py'
       f' --processed-dir "{SPLITS_ROOT}"'
       f' --variants-dir "{VARIANTS_DIR}"'
       f' --reference-dir "{REFERENCE_DIR}"'
       f' --out-dir "{FP_DIR}"'
       f' --datasets half_life_obach --resume')
print(cmd, "\n")
!{cmd}

cd /content/a3 && python -u score_variants_fingerprint.py --processed-dir "/content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo" --variants-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/variants_v2" --reference-dir "/content/drive/MyDrive/Conference_2026/Jiye/outputs" --out-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/fingerprint" --datasets half_life_obach --resume 

[1/1] half_life_obach: 원본 665 + 변형 5,931 채점 (22.9초, 대표 rf, 원본 대비 최대차 392.347)

완료: 1종, 변형 5,931건 채점
담당2 원본 예측과의 최대차 평균 392.3475 / 최대 392.3475


## 3단계 — 언어모델 채점

체크포인트는 **seed_42** 다. 기존 채점을 재현하는 시드임을 확인해 뒀다
(평균절대차 2e-8, 43은 0.0155, 44는 0.0131).

In [7]:
# ============================================================================
# [셀 6] ChemBERTa 채점 (진행 상황 실시간 출력)
# ============================================================================
cmd = (f'cd {WORK} && python score_variants_chemberta.py'
       f' --variants-dir "{VARIANTS_DIR}"'
       f' --splits-dir "{SPLITS_ROOT}"'
       f' --checkpoint-root "{CHECKPOINT_ROOT}"'
       f' --out-dir "{CB_DIR}"'
       f' --device cuda --batch-size 128 --resume')
print(cmd, "\n")
!{cmd}

cd /content/a3 && python score_variants_chemberta.py --variants-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/variants_v2" --splits-dir "/content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo" --checkpoint-root "/content/drive/MyDrive/Conference_2026/Jiye/checkpoints/chemberta_seed_42/checkpoints" --out-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/chemberta" --device cuda --batch-size 128 --resume 

장치: cuda
Loading weights: 100% 57/57 [00:05<00:00,  9.95it/s]
Loading weights: 100% 57/57 [00:00<00:00, 6221.22it/s]
Loading weights: 100% 57/57 [00:01<00:00, 30.27it/s]
Loading weights: 100% 57/57 [00:00<00:00, 2682.86it/s]
[1/22] ames: 변형 50,446 + 원본 7,255 × 2버전 (51.9초)
Loading weights: 100% 57/57 [00:00<00:00, 105.79it/s]
Loading weights: 100% 57/57 [00:00<00:00, 2485.81it/s]
Loading weights: 100% 57/57 [00:00<00:00, 80.42it/s]
Loading weights: 100% 57/57 [00:00<00:00, 2741.44it/s]
[2/22] bbb_martins: 변형 15,655 + 원본 1,955 × 2버전 (23.4초)
Load

## 채점 검증

두 가지를 본다.

1. 행 수와 결측
2. **A축 지문 분산이 0인가** — Morgan 지문은 SMILES 순서에 불변이므로 0이어야 한다.
   0이 아니면 A축 변형에 다른 이성질체가 섞인 것이고, 정준형 왕복 검사가
   제대로 작동하지 않았다는 뜻이다

In [8]:
# ============================================================================
# [셀 7] 채점 검증
# ============================================================================
rows = []
for ds in datasets:
    src = VARIANTS_DIR / ds / "variants.csv"
    n_in = sum(1 for _ in open(src, encoding="utf-8")) - 1
    rec = {"dataset": ds, "입력": n_in}

    for label, d, fname in [("지문", FP_DIR, None), ("CB", CB_DIR, None)]:
        hit = list((d / ds).glob("variant_predictions*.csv")) if (d / ds).exists() else []
        if not hit:
            rec[label] = "없음"; continue
        v = pd.read_csv(hit[0])
        pcols = [c for c in v.columns if c.startswith("pred")]
        rec[label] = len(v)
        rec[label + "_결측"] = int(v[pcols].isna().sum().sum())
        if label == "지문" and "axis" in v.columns and pcols:
            a = v[v.axis == "A"]
            if len(a):
                s = a.groupby("parent_row_uid")[pcols[0]].std(ddof=0)
                rec["A축_최대분산"] = float(s.max())
    rows.append(rec)

chk = pd.DataFrame(rows)
print(chk.to_string(index=False))

if "A축_최대분산" in chk.columns:
    m = chk["A축_최대분산"].max()
    print(f"\nA축 지문 분산 최댓값: {m:.3e}",
          "-> 0 확인. 왕복 검사 정상" if m < 1e-12 else "-> !! 0이 아니다. A축 변형 점검 필요")

                       dataset     입력     지문  지문_결측  A축_최대분산     CB  CB_결측
                          ames  50446  50446      0      0.0  50446      0
                   bbb_martins  15655  15655      0      0.0  15655      0
            bioavailability_ma   5222   5222      0      0.0   5222      0
                    caco2_wang   7960   7960      0      0.0   7960      0
       clearance_hepatocyte_az   8570   8570      0      0.0   8570      0
        clearance_microsome_az   9314   9314      0      0.0   9314      0
cyp2c9_substrate_carbonmangels   5589   5589      0      0.0   5589      0
                  cyp2c9_veith  97029  97029      0      0.0  97029      0
cyp2d6_substrate_carbonmangels   5407   5407      0      0.0   5407      0
                  cyp2d6_veith 105921 105921      0      0.0 105921      0
cyp3a4_substrate_carbonmangels   5579   5579      0      0.0   5579      0
                  cyp3a4_veith  99427  99427      0      0.0  99427      0
                         

In [13]:
# B가 0이 아닌지 추가 확인
rows = []
for ds in datasets:
    for label, d in [("fp", FP_DIR), ("cb", CB_DIR)]:
        f = list((d / ds).glob("variant_predictions*.csv"))[0]
        v = pd.read_csv(f)
        pcol = [c for c in v.columns if c.startswith("pred")][0]
        s = (v.groupby(["axis", "parent_row_uid"])[pcol].std(ddof=0)
             .groupby("axis").mean())
        rows.append({"dataset": ds, "모델": label, **s.round(5).to_dict()})

print(pd.DataFrame(rows).to_string(index=False))

                       dataset 모델       A  B1_protonation  B1_tautomer  B3_stereo
                          ames fp 0.00000         0.02209      0.02288    0.00821
                          ames cb 0.03770         0.00392      0.01161    0.00086
                   bbb_martins fp 0.00000         0.03194      0.02875    0.01535
                   bbb_martins cb 0.01599         0.00204      0.00445    0.00041
            bioavailability_ma fp 0.00000         0.02608      0.02268    0.01608
            bioavailability_ma cb 0.00636         0.00104      0.00245    0.00040
                    caco2_wang fp 0.00000         0.06329      0.05825    0.03334
                    caco2_wang cb 0.01415         0.00394      0.00703    0.00069
       clearance_hepatocyte_az fp 0.00000         4.29927      3.89716    0.72467
       clearance_hepatocyte_az cb 0.70035         0.16984      0.48500    0.01221
        clearance_microsome_az fp 0.00000         3.86852      3.72908    0.94057
        clearanc

## 4~8단계 — 인자 확인

여기서부터는 스크립트 인자를 모른다. `--help` 를 찍어 확인한다.

**8단계 `run_preregistered_ablation.py` 는 담당4의 로컬 절대경로가 하드코딩돼 있어
그대로 실행되지 않는다.** 경로 설정부를 함께 출력하니 고칠 위치를 확인한다.

In [12]:
# ============================================================================
# [셀 8] 4~8단계 스크립트 인자 확인
# ============================================================================
LATER = ["compute_ab_variance.py", "assemble_signals.py",
         "build_conditional_signals.py", "build_rich_variant_features.py",
         "run_preregistered_ablation.py"]

for name in LATER:
    p = WORK / name
    print("=" * 70)
    print(name)
    print("=" * 70)
    if not p.exists():
        print("  파일 없음\n"); continue
    r = subprocess.run([sys.executable, str(p), "--help"], capture_output=True, text=True)
    out = (r.stdout or r.stderr).strip()
    print(out[:1500] if out else "  (--help 없음 — argparse 미사용)")
    # 경로 하드코딩 여부
    hard = [l.strip() for l in p.read_text(encoding="utf-8").splitlines()
            if re.search(r'Path\(\s*["\']/|=\s*["\']/(Users|home|content)', l)]
    if hard:
        print("\n  [하드코딩 경로]")
        for l in hard[:6]:
            print("   ", l[:110])
    print()

compute_ab_variance.py
usage: compute_ab_variance.py [-h] --splits-dir SPLITS_DIR
                              --scores-dir SCORES_DIR --out-dir OUT_DIR
                              [--datasets [DATASETS ...]]

A축·B축 분산 신호 산출

options:
  -h, --help            show this help message and exit
  --splits-dir SPLITS_DIR
  --scores-dir SCORES_DIR
                        fingerprint·chemberta 상위
  --out-dir OUT_DIR
  --datasets [DATASETS ...]

assemble_signals.py
usage: assemble_signals.py [-h] --jiye-dir JIYE_DIR --scores-dir SCORES_DIR
                           --out-dir OUT_DIR [--datasets [DATASETS ...]]

기준선과 A·B 신호 통합

options:
  -h, --help            show this help message and exit
  --jiye-dir JIYE_DIR   담당2 outputs 최상위
  --scores-dir SCORES_DIR
                        signals·fp_conformal 상위
  --out-dir OUT_DIR
  --datasets [DATASETS ...]

build_conditional_signals.py
usage: build_conditional_signals.py [-h] --scores-dir SCORES_DIR
                                    --variants-d

## 다음

셀 8 출력을 공유하면 4~8단계 셀을 완성한다. 확인할 것은 셋이다.

- 각 스크립트가 받는 인자 이름
- 8단계의 하드코딩 경로 위치 (고칠 지점)
- 4단계가 지문·언어모델 채점 결과를 각각 어떤 인자로 받는지

## 판정 기준 (8단계 이후)

| 기준 | 현재 | 목표 |
|---|---|---|
| 축 기여 (AUPRC) | +0.0225 | 초과 |
| B3 기여 (정규화 AURC) | 0.984 (거의 무작위) | 0에서 이탈 |

전후 수치를 함께 보고한다 (협업 규칙).

## 산출물

```
Yoonsoo/scores_v2/
  fingerprint/   지문 채점
  chemberta/     언어모델 채점
  signals/       축 분산
  evaluation/    신호 통합 + 조건부 B + 확장 통계
```

기존 `scores_role4` 는 건드리지 않는다.

In [14]:
# ============================================================================
# [셀 9] 5단계 준비 — fp_conformal 확보
# ----------------------------------------------------------------------------
# assemble_signals.py 는 --scores-dir 아래 signals/ 와 fp_conformal/ 을 찾는다.
# fp_conformal 은 원본 분자와 splits 기반이고 변형과 무관하므로 기존 것을 재사용한다.
# ============================================================================
FPC_DIR = SCORES / "fp_conformal"

CANDS = [TEAM / "Juhyeong" / "data" / "processed" / "scores_role4" / "fp_conformal",
         TEAM / "Main" / "fp_conformal_role4"]
src = next((p for p in CANDS if p.exists()), None)
print("원본:", src or "!! 못 찾음")
assert src, f"fp_conformal 을 못 찾았다. 후보: {[str(p) for p in CANDS]}"

if not FPC_DIR.exists():
    shutil.copytree(src, FPC_DIR, ignore=shutil.ignore_patterns("MANIFEST.csv"))
print("복사 완료:", FPC_DIR)
print("물성:", len([p for p in FPC_DIR.iterdir() if p.is_dir() and not p.name.startswith("_")]))

원본: /content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/scores_role4/fp_conformal
복사 완료: /content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/fp_conformal
물성: 22


In [15]:
# ============================================================================
# [셀 10] 4단계 — 축 분산 (흔들림 계산)
# ============================================================================
cmd = (f'cd {WORK} && python -u compute_ab_variance.py'
       f' --splits-dir "{SPLITS_ROOT}"'
       f' --scores-dir "{SCORES}"'
       f' --out-dir "{SIG_DIR}"')
print(cmd, "\n")
!{cmd}

cd /content/a3 && python -u compute_ab_variance.py --splits-dir "/content/drive/MyDrive/Conference_2026/Juhyeong/data/processed/pipeline_yoonsoo" --scores-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2" --out-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/signals" 

[1/22] ames: 1,450행
[2/22] bbb_martins: 390행
[3/22] bioavailability_ma: 127행
[4/22] caco2_wang: 180행
[5/22] clearance_hepatocyte_az: 203행
[6/22] clearance_microsome_az: 220행
[7/22] cyp2c9_substrate_carbonmangels: 133행
[8/22] cyp2c9_veith: 2,409행
[9/22] cyp2d6_substrate_carbonmangels: 132행
[10/22] cyp2d6_veith: 2,616행
[11/22] cyp3a4_substrate_carbonmangels: 133행
[12/22] cyp3a4_veith: 2,460행
[13/22] dili: 94행
[14/22] half_life_obach: 132행
[15/22] herg: 127행
[16/22] hia_hou: 116행
[17/22] ld50_zhu: 1,468행
[18/22] lipophilicity_astrazeneca: 838행
[19/22] pgp_broccatelli: 242행
[20/22] ppbr_az: 358행
[21/22] solubility_aqsoldb: 1,915행
[22/22] vdss_lombardo: 222행

=== 축·모델별 신호 크기와 오차 상관 (test, 22종 중앙값) =

In [19]:
# ============================================================================
# [셀 14] 8단계 — 판정 (경로만 패치)
# ----------------------------------------------------------------------------
# 계산 로직은 손대지 않는다. 담당4 로컬 절대경로 3곳만 우리 것으로 바꾼다.
# ============================================================================
import re

ABL_DIR = SCORES / "ablation"
ABL_DIR.mkdir(parents=True, exist_ok=True)

src_txt = (WORK / "run_preregistered_ablation.py").read_text(encoding="utf-8")
orig = src_txt

src_txt = re.sub(r'^B\s*=\s*Path\(.*\)\s*$', f'B=Path(r"{TEAM}")',
                 src_txt, count=1, flags=re.M)
src_txt = re.sub(r'^EV\s*=\s*B\s*/.*$',      f'EV=Path(r"{EVAL_DIR}")',
                 src_txt, count=1, flags=re.M)
src_txt = re.sub(r'B\s*/\s*"Juhyeong[^"]*preregistered_ablation\.csv"',
                 f'Path(r"{ABL_DIR / "preregistered_ablation.csv"}")', src_txt)

patched = WORK / "run_preregistered_ablation_patched.py"
patched.write_text(src_txt, encoding="utf-8")

print("=== 바뀐 줄 ===")
for a, b in zip(orig.splitlines(), src_txt.splitlines()):
    if a != b:
        print("-", a[:110])
        print("+", b[:110])

assert "zzuhyeong2" not in src_txt, "하드코딩 경로가 남아 있다"
compile(src_txt, "patched", "exec")
print("\n패치 확인 완료")

=== 바뀐 줄 ===
- B=Path("/Users/zzuhyeong2/Library/CloudStorage/GoogleDrive-a01056371120@gmail.com/My Drive/Conference_2026")
+ B=Path(r"/content/drive/MyDrive/Conference_2026")
- EV=B/"Juhyeong/data/processed/scores_role4/evaluation"
+ EV=Path(r"/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/evaluation")
- t.to_csv(B/"Juhyeong/data/processed/scores_role4/expanded_ablation_22/preregistered_ablation.csv",index=False)
+ t.to_csv(Path(r"/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/ablation/preregistered_ablation.csv")

패치 확인 완료


In [26]:
# ============================================================================
# [교체] reports 를 우리 공간으로 복사하고 확정판 적용
# ----------------------------------------------------------------------------
# 담당4 폴더는 건드리지 않는다. 우리 사본에서만 허용성 표를 확정판으로 바꾼다.
# ============================================================================
MY_REPORTS = SCORES / "reports"
FINAL = REPORTS_DIR / "06_transformation_allowance_final.csv"

if MY_REPORTS.exists():
    shutil.rmtree(MY_REPORTS)
shutil.copytree(REPORTS_DIR, MY_REPORTS)
print("reports 사본:", MY_REPORTS)

# 스크립트가 읽는 이름으로 확정판을 넣는다 (사본에서만)
target = MY_REPORTS / "06_transformation_allowance_revised.csv"
shutil.copy2(FINAL, target)

chk = pd.read_csv(target)
print("\nB1_protonation:", chk.B1_protonation.value_counts().to_dict())
print(chk.loc[chk.dataset.str.contains("substrate"), ["dataset", "B1_protonation"]].to_string(index=False))
print("\n07_axis_decision 있음:", (MY_REPORTS / "07_axis_decision.csv").exists())

reports 사본: /content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/reports

B1_protonation: {'허용': 12, '주의': 10}
                       dataset B1_protonation
cyp2c9_substrate_carbonmangels             주의
cyp2d6_substrate_carbonmangels             주의
cyp3a4_substrate_carbonmangels             주의

07_axis_decision 있음: True


In [30]:
# ============================================================================
# [재실행] 5 -> 6 -> 7 -> 8단계 일괄 수행
# ============================================================================
if EVAL_DIR.exists():
    shutil.rmtree(EVAL_DIR)          # 깨끗하게 다시 만든다
    print("기존 evaluation 제거\n")

for label, cmd in [
    ("5 신호 통합", f'python -u assemble_signals.py --jiye-dir "{REFERENCE_DIR}"'
                    f' --scores-dir "{SCORES}" --out-dir "{EVAL_DIR}"'),
    ("6 조건부 B",  f'python -u build_conditional_signals.py --scores-dir "{SCORES}"'
                    f' --variants-dir "{VARIANTS_DIR}" --reports-dir "{MY_REPORTS}"'
                    f' --evaluation-dir "{EVAL_DIR}"'),
    ("7 확장 통계", f'python -u build_rich_variant_features.py --scores-dir "{SCORES}"'
                    f' --reports-dir "{MY_REPORTS}" --evaluation-dir "{EVAL_DIR}"'),
    ("8 판정",      f'python -u run_preregistered_ablation_patched.py'),
]:
    print("=" * 60); print(label); print("=" * 60)
    !cd {WORK} && {cmd}
    print()

기존 evaluation 제거

5 신호 통합
[1/22] ames: 1,450행
[2/22] bbb_martins: 390행
[3/22] bioavailability_ma: 127행
[4/22] caco2_wang: 180행
[5/22] clearance_hepatocyte_az: 203행
[6/22] clearance_microsome_az: 220행
[7/22] cyp2c9_substrate_carbonmangels: 133행
[8/22] cyp2c9_veith: 2,409행
[9/22] cyp2d6_substrate_carbonmangels: 132행
[10/22] cyp2d6_veith: 2,616행
[11/22] cyp3a4_substrate_carbonmangels: 133행
[12/22] cyp3a4_veith: 2,460행
[13/22] dili: 94행
[14/22] half_life_obach: 132행
[15/22] herg: 127행
[16/22] hia_hou: 116행
[17/22] ld50_zhu: 1,468행
[18/22] lipophilicity_astrazeneca: 838행
[19/22] pgp_broccatelli: 242행
[20/22] ppbr_az: 358행
[21/22] solubility_aqsoldb: 1,915행
[22/22] vdss_lombardo: 222행

=== 신호별 오차 상관 (지문 대표 모델 기준, 22종 중앙값) ===
group                             signal       중앙  양수  유효
   기준                 base__conformal_fp 0.314230  21  22
   기준                       base__ad_knn 0.163835  19  22
   기준                   base__ad_density 0.157963  21  22
   기준                 base__conformal_

In [31]:
# ============================================================================
# [셀 16] half_life_obach 포함/제외 비교
# ============================================================================
from scipy.stats import wilcoxon

t = pd.read_csv(ABL_DIR / "preregistered_ablation.csv")
base = "기준(AD+컨포멀)"
cfgs = [c[len("auprc__"):] for c in t.columns if c.startswith("auprc__") and c[len("auprc__"):] != base]

for label, d in [("22종", t), ("21종 (half_life 제외)", t[t.dataset != "half_life_obach"])]:
    print(f"\n=== {label}  n={len(d)} ===")
    for n in cfgs:
        for met, sign in (("auprc", 1), ("aurc", -1)):
            x = ((d[f"{met}__{n}"] - d[f"{met}__{base}"]) * sign).dropna()
            try:    p = f"{wilcoxon(x)[1]:.3f}"
            except Exception: p = "-"
            print(f"  {n:22s} {met.upper():5s} {x.mean():+.4f}  개선 {int((x>0).sum())}/{len(x)}  p={p}")


=== 22종  n=22 ===
  적용가능도메인 단독             AUPRC -0.0425  개선 5/22  p=0.004
  적용가능도메인 단독             AURC  -0.1506  개선 4/22  p=0.001
  기준+모델불일치               AUPRC +0.0475  개선 16/22  p=0.007
  기준+모델불일치               AURC  +0.0718  개선 17/22  p=0.004
  기준+A                   AUPRC +0.0081  개선 14/22  p=0.248
  기준+A                   AURC  +0.0106  개선 13/22  p=0.222
  기준+B                   AUPRC +0.0009  개선 11/22  p=0.903
  기준+B                   AURC  +0.0242  개선 13/22  p=0.063
  기준+A+B                 AUPRC +0.0093  개선 14/22  p=0.276
  기준+A+B                 AURC  +0.0169  개선 16/22  p=0.079
  전체                     AUPRC +0.0504  개선 18/22  p=0.001
  전체                     AURC  +0.0836  개선 17/22  p=0.001

=== 21종 (half_life 제외)  n=21 ===
  적용가능도메인 단독             AUPRC -0.0405  개선 5/21  p=0.008
  적용가능도메인 단독             AURC  -0.1606  개선 3/21  p=0.000
  기준+모델불일치               AUPRC +0.0461  개선 15/21  p=0.011
  기준+모델불일치               AURC  +0.0769  개선 17/21  p=0.002
  기준+A                 

In [32]:
# ============================================================================
# [추가] 확장 통계 기여 비교 — 스크립트 확보 + 인자 확인
# ============================================================================
for name in ("run_rich_feature_comparison.py", "compare_signal_contributions.py"):
    src = TEAM / "Main" / "scripts_role4" / name
    if src.exists():
        shutil.copy2(src, WORK / name)
        print(f"[복사] {name}")
    else:
        print(f"[없음] {name}")

print()
for name in ("run_rich_feature_comparison.py", "compare_signal_contributions.py"):
    p = WORK / name
    if not p.exists():
        continue
    print("=" * 70); print(name); print("=" * 70)
    r = subprocess.run([sys.executable, str(p), "--help"], capture_output=True, text=True)
    print((r.stdout or r.stderr)[:1200])
    hard = [l.strip() for l in p.read_text(encoding="utf-8").splitlines()
            if re.search(r'Path\(\s*["\']/|=\s*["\']/(Users|home|content)', l)]
    if hard:
        print("  [하드코딩 경로]")
        for l in hard[:5]:
            print("   ", l[:110])
    print()

[복사] run_rich_feature_comparison.py
[복사] compare_signal_contributions.py

run_rich_feature_comparison.py
Traceback (most recent call last):
  File "/content/a3/run_rich_feature_comparison.py", line 43, in <module>
    for n,l,f in cfg: out[n]=run(l,f)
                             ~~~^^^^^
  File "/content/a3/run_rich_feature_comparison.py", line 19, in run
    for d in sorted(p.name for p in EV.iterdir() if p.is_dir() and not p.name.startswith("_")):
                                    ~~~~~~~~~~^^
  File "/usr/lib/python3.13/pathlib/_local.py", line 575, in iterdir
    with os.scandir(root_dir) as scandir_it:
         ~~~~~~~~~~^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/Users/zzuhyeong2/Library/CloudStorage/GoogleDrive-a01056371120@gmail.com/My Drive/Conference_2026/Juhyeong/data/processed/scores_role4/evaluation'

  [하드코딩 경로]
    B=Path("/Users/zzuhyeong2/Library/CloudStorage/GoogleDrive-a01056371120@gmail.com/My Drive/Conference_2026")

compare_signal_contr

In [33]:
# ============================================================================
# [추가] run_rich_feature_comparison.py 경로 패치
# ----------------------------------------------------------------------------
# 계산 로직은 손대지 않는다. scores_role4 로 향하는 출력도 우리 scores_v2 로 옮긴다.
# ============================================================================
p = WORK / "run_rich_feature_comparison.py"
src = p.read_text(encoding="utf-8")
orig = src

src = re.sub(r'^B\s*=\s*Path\(.*\)\s*$', f'B=Path(r"{TEAM}")', src, count=1, flags=re.M)
src = re.sub(r'^EV\s*=\s*B\s*/.*$',      f'EV=Path(r"{EVAL_DIR}")', src, count=1, flags=re.M)

def _repl(m):
    t = SCORES / m.group(1)                      # 같은 하위 경로를 우리 쪽으로
    t.parent.mkdir(parents=True, exist_ok=True)
    return f'Path(r"{t}")'

src = re.sub(r'B\s*/\s*"Juhyeong/data/processed/scores_role4/([^"]+)"', _repl, src)

patched = WORK / "run_rich_feature_comparison_patched.py"
patched.write_text(src, encoding="utf-8")

print("=== 바뀐 줄 ===")
for a, b in zip(orig.splitlines(), src.splitlines()):
    if a != b:
        print("-", a[:110]); print("+", b[:110])

for bad in ("zzuhyeong2", "scores_role4"):
    assert bad not in src, f"{bad} 가 남아 있다"
compile(src, "patched", "exec")
print("\n패치 확인 완료")

=== 바뀐 줄 ===
- B=Path("/Users/zzuhyeong2/Library/CloudStorage/GoogleDrive-a01056371120@gmail.com/My Drive/Conference_2026")
+ B=Path(r"/content/drive/MyDrive/Conference_2026")
- EV=B/"Juhyeong/data/processed/scores_role4/evaluation"
+ EV=Path(r"/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/evaluation")

패치 확인 완료


In [34]:
# ============================================================================
# [추가] 확장 통계 기여 비교 실행  — 축 기여 +0.0225 대조
# ============================================================================
!cd {WORK} && python -u run_rich_feature_comparison_patched.py

구성                        AUPRC     AURC
Ridge / 기준만              0.3712   0.5348
Ridge / +단순축             0.3741   0.5229
Ridge / +확장축             0.3798   0.5194
로지스틱 / 기준만               0.3671   0.5268
로지스틱 / +단순축              0.3735   0.5396
로지스틱 / +확장축              0.3873   0.5920

=== 축 추가 효과 (같은 학습기 안에서 기준만 대비) ===
  Ridge / +단순축           AUPRC +0.0028 14/22종 p=0.156   AURC +0.0118 12/22종 p=0.248
  Ridge / +확장축           AUPRC +0.0086 13/22종 p=0.443   AURC +0.0153 13/22종 p=0.483
  로지스틱 / +단순축            AUPRC +0.0063 12/22종 p=0.406   AURC -0.0128 13/22종 p=0.443
  로지스틱 / +확장축            AUPRC +0.0201 12/22종 p=0.483   AURC -0.0652 9/22종 p=0.545


In [35]:
# ============================================================================
# [추가] 신호 기여도 동일 잣대 비교
# ============================================================================
CONTRIB_DIR = SCORES / "contribution"
cmd = (f'cd {WORK} && python -u compare_signal_contributions.py'
       f' --evaluation-dir "{EVAL_DIR}"'
       f' --out-dir "{CONTRIB_DIR}"')
print(cmd, "\n")
!{cmd}

cd /content/a3 && python -u compare_signal_contributions.py --evaluation-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/evaluation" --out-dir "/content/drive/MyDrive/Conference_2026/Yoonsoo/scores_v2/contribution" 

=== 신호 기여도 동일 잣대 비교 (기준선은 제거 손실, 제안 축은 추가 이득) ===
구분         신호                              AUPRC     물성      p     AURC     물성      p
기존 기준선     모델 불일치                        +0.0475   16/22  0.007  +0.0718   17/22  0.004
기존 기준선     컨포멀 (지문)                      +0.0049   12/22  0.545  +0.0679   18/22  0.001
기존 기준선     컨포멀 (ChemBERTa)               +0.0029   13/22  0.821  +0.0083   14/22  0.156
기존 기준선     적용가능도메인 이웃 유사도                +0.0017   10/22  0.455  +0.0102   15/22  0.079
기존 기준선     적용가능도메인 국소 밀도                 -0.0060    6/22  0.276  -0.0001   10/22  0.726
제안 축       입력 상태 민감성 B (조건부)             +0.0038   14/22  0.192  +0.0119   10/22  0.498
제안 축       표현 불안정성 A                     +0.0035    9/22  0.821  +0.0085   16/22  0.073
제안 축       A + B 

In [36]:
# ============================================================================
# [확인] cond_B 가 상수인 물성 + 물성별 B 효과
# ============================================================================
const_ds = []
for ds in datasets:
    e = pd.read_csv(EVAL_DIR / ds / "evaluation_signals.csv", low_memory=False)
    c = [x for x in e.columns if x.startswith("cond_B") and not x.endswith("__pct")]
    bad = [x for x in c if e[x].nunique() <= 1]
    if bad:
        const_ds.append(ds)
        print(f"{ds:32s} 상수 {len(bad)}/{len(c)}  값 {e[bad[0]].iloc[0]}")
print("\n상수 물성:", const_ds or "없음")

t = pd.read_csv(ABL_DIR / "preregistered_ablation.csv")
base = "기준(AD+컨포멀)"
t["dB_auprc"] = t[f"auprc__기준+B"] - t[f"auprc__{base}"]
t["dB_aurc"]  = t[f"aurc__{base}"]  - t[f"aurc__기준+B"]
print("\n=== 물성별 B 효과 (AURC 오름차순) ===")
print(t[["dataset", "dB_auprc", "dB_aurc"]].round(4).sort_values("dB_aurc").to_string(index=False))

print("\n=== 평균 비교 ===")
for label, d in [("22종 전체", t),
                 ("상수 물성 제외", t[~t.dataset.isin(const_ds)])]:
    print(f"  {label:16s} n={len(d):2d}  AUPRC {d.dB_auprc.mean():+.4f}  AURC {d.dB_aurc.mean():+.4f}")

solubility_aqsoldb               상수 2/2  값 0.0

상수 물성: ['solubility_aqsoldb']

=== 물성별 B 효과 (AURC 오름차순) ===
                       dataset  dB_auprc  dB_aurc
                    caco2_wang   -0.0075  -0.0431
                          herg   -0.0400  -0.0323
                      ld50_zhu   -0.0022  -0.0287
     lipophilicity_astrazeneca    0.0038  -0.0183
                          dili   -0.0200  -0.0092
                          ames    0.0105  -0.0083
               pgp_broccatelli   -0.0291  -0.0073
                   bbb_martins    0.0743  -0.0016
            solubility_aqsoldb    0.0000   0.0000
               half_life_obach   -0.0081   0.0043
                       hia_hou   -0.0393   0.0100
                       ppbr_az    0.0101   0.0158
cyp2d6_substrate_carbonmangels    0.0160   0.0209
                 vdss_lombardo    0.0252   0.0229
        clearance_microsome_az    0.0062   0.0255
cyp3a4_substrate_carbonmangels    0.0084   0.0300
                  cyp3a4_veith    0.0164  

In [37]:
# ============================================================================
# [pH 창 검증] 담당4 물성별 결과와 대조
# ============================================================================
old = pd.read_csv(TEAM / "Main" / "expanded_ablation_22_role4" / "preregistered_ablation.csv")
new = pd.read_csv(ABL_DIR / "preregistered_ablation.csv")
base = "기준(AD+컨포멀)"

def dB(df):
    return pd.DataFrame({
        "dataset": df.dataset,
        "auprc": df[f"auprc__기준+B"] - df[f"auprc__{base}"],
        "aurc":  df[f"aurc__{base}"]  - df[f"aurc__기준+B"]})

m = dB(old).merge(dB(new), on="dataset", suffixes=("_old", "_new"))
m["d_aurc"] = (m.aurc_new - m.aurc_old).round(4)

# 구판 허용성 기준 (담당4 실행 당시): protonation 주의 7종
CAUTION_OLD = ["caco2_wang", "solubility_aqsoldb", "ld50_zhu", "hia_hou",
               "bioavailability_ma", "ames", "dili"]
SUBSTRATE = [d for d in m.dataset if "substrate" in d]      # 판본 차이로 오염됨

m["group"] = np.where(m.dataset.isin(SUBSTRATE), "판본차이",
                np.where(m.dataset.isin(CAUTION_OLD), "주의(pH 무관)", "허용(pH 영향)"))

print(m[["dataset", "group", "aurc_old", "aurc_new", "d_aurc"]].round(4).to_string(index=False))
print("\n=== 그룹별 AURC 변화 (우리 - 담당4) ===")
print(m.groupby("group").d_aurc.agg(["count", "median", "mean"]).round(4).to_string())

                       dataset     group  aurc_old  aurc_new  d_aurc
                          ames 주의(pH 무관)   -0.0078   -0.0083 -0.0005
                   bbb_martins 허용(pH 영향)    0.0098   -0.0016 -0.0114
            bioavailability_ma 주의(pH 무관)    0.1113    0.0838 -0.0274
                    caco2_wang 주의(pH 무관)   -0.0710   -0.0431  0.0279
       clearance_hepatocyte_az 허용(pH 영향)    0.0845    0.0811 -0.0034
        clearance_microsome_az 허용(pH 영향)    0.0055    0.0255  0.0199
cyp2c9_substrate_carbonmangels      판본차이    0.1373    0.2019  0.0647
                  cyp2c9_veith 허용(pH 영향)    0.0567    0.0646  0.0079
cyp2d6_substrate_carbonmangels      판본차이    0.1075    0.0209 -0.0866
                  cyp2d6_veith 허용(pH 영향)    0.0671    0.0752  0.0081
cyp3a4_substrate_carbonmangels      판본차이    0.0342    0.0300 -0.0042
                  cyp3a4_veith 허용(pH 영향)    0.0438    0.0459  0.0021
                          dili 주의(pH 무관)   -0.0156   -0.0092  0.0064
               half_life_obach 허용(